# 08 · 3D map for ArcGIS Pro / QGIS

The frozen final model predicts every 0.5 m cell (0-15 m below the DEM surface) under a **50 m grid** that covers the area of `settings.toml` (Downtown, or Downtown + 5 km ring).

| Column | Meaning |
|---|---|
| `X, Y, Z` | UTM 48N (EPSG:32648) and the cell-centre elevation |
| `p_clay`, `clay` | probability of Clay, and the class at the final model's threshold (1 = Clay) |
| `dist_nearest_bh_m`, `reliability_zone` | distance to the nearest training borehole; zone 1 = ≤ 100 m, 2 = 100-300 m, 3 = > 300 m. Prediction skill falls with distance from boreholes (notebook 04), so show this zone with the map |
| `extrap_vertical` | 1 = cell elevation outside the elevations of the training data |

**ArcGIS Pro:** *XY Table To Point* on the CSV, X = `X`, Y = `Y`, Z = `Z`, coordinate system EPSG:32648. For a voxel layer, add the `.nc` file (*Add Multidimensional Voxel Layer*).

**Run time:** 5-15 min for Downtown, about 4× longer for Downtown + 5 km (about 4 million cells).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])

model, rec = H.load_final(S)
fin = rec
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
SPACING = S["quick"]["map_spacing_m"] if S["run"]["quick"] else S["map"]["spacing_m"]
OUT, PRIV = H.out_dir(S, "08_map"), H.out_dir(S, "08_map", private=True)
print("Model:", fin["model"], "|", fin["features"], "| threshold", fin["threshold"], "| grid", SPACING, "m | area", S["area"]["area"])

In [ ]:
cols = H.map_columns(S, SPACING)
grid = H.map_cells(cols, S)
print(f"{len(cols)} grid columns, {len(grid)} cells")
n = S["map"]["chunk_rows"]
p = np.concatenate([model.p_clay(grid.iloc[i:i + n]) for i in range(0, len(grid), n)]).astype("float32")
grid["p_clay"] = p
grid["clay"] = (p >= fin["threshold"]).astype("int8")
bh_xy = cells.groupby("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].first().values
d = H.nearest_distance(cols[["X", "Y"]].values, bh_xy)
grid["dist_nearest_bh_m"] = d[grid.col].astype("float32")
grid["reliability_zone"] = (np.searchsorted(S["map"]["zone_edges_m"], grid.dist_nearest_bh_m, side="right") + 1).astype("int8")
lo, hi = rec["top_elevation_range_m"]
grid["extrap_vertical"] = ((grid.Top_Elevation_m < lo) | (grid.Top_Elevation_m > hi)).astype("int8")

In [ ]:
tag = f"{S['area']['area']}_{int(SPACING)}m"
grid.rename(columns={"UTM_Easting_m": "X", "UTM_Northing_m": "Y"})[
    ["X", "Y", "Z", "depth_top_m", "Top_Elevation_m", "ground_m", "p_clay", "clay", "dist_nearest_bh_m",
     "reliability_zone", "extrap_vertical"]].to_csv(PRIV / f"map_3d_points_{tag}.csv", index=False, float_format="%.3f")
vox = H.to_voxel_netcdf(grid, cols, S, PRIV / f"map_3d_voxels_{tag}.nc")
summary = {"model": fin["model"], "model_sha256": fin["sha256"], "area": S["area"]["area"], "spacing_m": SPACING,
           "columns": len(cols), "cells": len(grid), "predicted_clay_share": float(grid.clay.mean()),
           "reliability_zone_share": grid.reliability_zone.value_counts(normalize=True).sort_index().round(4).to_dict(),
           "vertical_extrapolation_share": float(grid.extrap_vertical.mean()), "voxels": vox}
(OUT / f"map_summary_{tag}.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str))

## Quick look: four depths

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True)
for ax, depth in zip(axes, [1.0, 5.0, 10.0, 14.0]):
    g = grid[np.isclose(grid.depth_top_m, depth)]
    sc = ax.scatter(g.UTM_Easting_m, g.UTM_Northing_m, c=g.p_clay, s=1, cmap="RdYlBu_r", vmin=0, vmax=1)
    ax.set_title(f"P(Clay), {depth:g} m depth"); ax.set_aspect("equal"); ax.set_xticks([])
fig.colorbar(sc, ax=axes, fraction=0.02)
fig.savefig(PRIV / f"fig_map_depths_{tag}.png", dpi=150, bbox_inches="tight")
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 09.")